# Lecture 5: UMAP — Uniform Manifold Approximation and Projection

**Course: Foundations of AI with Embeddings**  
Constructor University, 2026

---

## Overview

UMAP (McInnes et al., 2018) is a nonlinear dimensionality reduction method grounded in Riemannian geometry and algebraic topology. Like t-SNE it preserves local structure, but it also better preserves global structure and is significantly faster.

**Key ideas:**
1. Model the high-dimensional data as a fuzzy topological structure (weighted graph).
2. Find a low-dimensional representation whose fuzzy topology is as similar as possible.

This notebook covers:
- Basic UMAP on MNIST digits — visual comparison with PCA and t-SNE
- Effect of key hyperparameters: `n_neighbors` and `min_dist`
- UMAP on synthetic high-dimensional embeddings (simulating word/document embeddings)
- Supervised UMAP using label information

In [ ]:
# Install required packages if needed
# !pip install umap-learn scikit-learn matplotlib seaborn

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.datasets import load_digits, make_blobs
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

import umap

# Reproducibility
SEED = 42
np.random.seed(SEED)

print("Libraries loaded successfully.")

---
## 1. UMAP Algorithm — Brief Recap

Given $N$ points $\{x_i\}_{i=1}^N$ in $\mathbb{R}^p$, UMAP proceeds in two phases:

### Phase 1 — Build the high-dimensional fuzzy graph

For each point $x_i$, find its $k$ nearest neighbours.  
Define a local metric that normalises distances so that the nearest neighbour is at distance 1:

$$p_{j|i} = \exp\!\left(\frac{-(d(x_i,x_j)-\rho_i)}{\sigma_i}\right)$$

where $\rho_i$ = distance to the nearest neighbour, and $\sigma_i$ is chosen so that $\sum_j p_{j|i} = \log_2 k$.  
Symmetrise: $w_{ij} = p_{j|i} + p_{i|j} - p_{j|i}\,p_{i|j}$.

### Phase 2 — Optimise the low-dimensional layout

Place points $\{y_i\}$ in $\mathbb{R}^q$ (usually $q=2$).  
Low-dimensional similarity:

$$q_{ij} = \left(1 + a\,\|y_i - y_j\|^{2b}\right)^{-1}$$

Parameters $a, b$ are determined by `min_dist`.  
Minimise the cross-entropy between the two fuzzy graphs:

$$\mathcal{L} = \sum_{i<j}\Bigl[w_{ij}\log\frac{w_{ij}}{q_{ij}} + (1-w_{ij})\log\frac{1-w_{ij}}{1-q_{ij}}\Bigr]$$

Optimised with stochastic gradient descent (negative sampling for efficiency).

### Key hyperparameters

| Parameter | Effect |
|---|---|
| `n_neighbors` | How many neighbours define the local neighbourhood. **Small** → focus on fine local structure; **large** → preserve more global structure. |
| `min_dist` | Minimum distance between points in the embedding. **Small** → tight clusters; **large** → more uniform spread. |
| `n_components` | Dimension of the target space. |
| `metric` | Distance metric in the input space (euclidean, cosine, …). |

---
## 2. MNIST Digits — PCA vs t-SNE vs UMAP

In [ ]:
# Load the sklearn digits dataset (8x8 images, 10 classes, 1797 samples)
digits = load_digits()
X = digits.data          # shape (1797, 64)
y = digits.target        # labels 0-9

# Standardise
X_scaled = StandardScaler().fit_transform(X)

print(f"Dataset: {X.shape[0]} samples, {X.shape[1]} features, {len(np.unique(y))} classes")

In [ ]:
# ---- PCA (2D) ----
pca = PCA(n_components=2, random_state=SEED)
X_pca = pca.fit_transform(X_scaled)

# ---- t-SNE (2D) ----
tsne = TSNE(n_components=2, perplexity=30, random_state=SEED, max_iter=1000)
X_tsne = tsne.fit_transform(X_scaled)

# ---- UMAP (2D) ----
reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=SEED)
X_umap = reducer.fit_transform(X_scaled)

print("Embeddings computed.")

In [ ]:
def plot_embedding(ax, Z, labels, title):
    """Scatter plot of a 2D embedding coloured by class label."""
    palette = sns.color_palette("tab10", 10)
    for cls in range(10):
        mask = labels == cls
        ax.scatter(Z[mask, 0], Z[mask, 1],
                   s=8, alpha=0.7, color=palette[cls], label=str(cls))
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_xticks([])
    ax.set_yticks([])


fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
plot_embedding(axes[0], X_pca,  y, "PCA")
plot_embedding(axes[1], X_tsne, y, "t-SNE  (perplexity=30)")
plot_embedding(axes[2], X_umap, y, "UMAP  (n_neighbors=15, min_dist=0.1)")

handles, labels_legend = axes[2].get_legend_handles_labels()
fig.legend(handles, labels_legend, title="Digit",
           loc="lower center", ncol=10, frameon=False,
           bbox_to_anchor=(0.5, -0.02))

fig.suptitle("MNIST digits: dimensionality reduction to 2D", fontsize=14)
plt.tight_layout()
plt.savefig("lectures/img/umap_vs_pca_tsne.png", dpi=120, bbox_inches="tight")
plt.show()

**Observations:**
- **PCA** is linear — it separates some digits but many clusters overlap.
- **t-SNE** reveals clear clusters but distorts global distances (inter-cluster gaps are not meaningful).
- **UMAP** produces similarly tight clusters while better preserving the overall topology (distances between clusters are more meaningful).

---
## 3. Effect of Hyperparameters

In [ ]:
# ---- Vary n_neighbors ----
n_neighbors_values = [5, 15, 50]
min_dist_values    = [0.0, 0.1, 0.5]

fig, axes = plt.subplots(3, 3, figsize=(13, 12))

for i, nn in enumerate(n_neighbors_values):
    for j, md in enumerate(min_dist_values):
        emb = umap.UMAP(
            n_neighbors=nn, min_dist=md,
            n_components=2, random_state=SEED
        ).fit_transform(X_scaled)

        ax = axes[i][j]
        palette = sns.color_palette("tab10", 10)
        for cls in range(10):
            mask = y == cls
            ax.scatter(emb[mask, 0], emb[mask, 1],
                       s=5, alpha=0.6, color=palette[cls])
        ax.set_title(f"n_neighbors={nn}, min_dist={md}", fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])

fig.suptitle("UMAP hyperparameter grid (MNIST digits)", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("lectures/img/umap_hyperparameters.png", dpi=120, bbox_inches="tight")
plt.show()

**Observations:**
- **Small `n_neighbors` (5):** very tight, fragmented clusters — captures fine local structure but may miss global patterns.
- **Large `n_neighbors` (50):** smoother, more connected layout — better global structure but less fine-grained separation.
- **Small `min_dist` (0.0):** points packed densely within each cluster.
- **Large `min_dist` (0.5):** points spread out within clusters, resembling a more uniform distribution.

---
## 4. UMAP on Synthetic High-Dimensional Embeddings

In practice, UMAP is frequently applied to word or document embedding vectors (e.g. Word2Vec, GloVe, BERT).
Here we simulate such a scenario by generating synthetic Gaussian clusters in a 50-dimensional space,
which mimics the structure of real embedding spaces.
Each cluster represents a semantic group (e.g. documents about the same topic).

In [ ]:
# Simulate 6 semantic clusters in a 50-dimensional embedding space
# (analogous to topic clusters in a document embedding space)
n_topics = 6
topic_names = [
    "science/space",
    "science/medicine",
    "computer graphics",
    "politics",
    "sports",
    "philosophy",
]

X_embed, y_embed = make_blobs(
    n_samples=600,
    n_features=50,        # 50-dimensional 'embedding' space
    centers=n_topics,
    cluster_std=3.5,
    random_state=SEED,
)

print(f"Synthetic embeddings: {X_embed.shape[0]} docs × {X_embed.shape[1]} dims, "
      f"{n_topics} topic clusters")

In [ ]:
# UMAP with Euclidean metric on the synthetic embeddings
reducer_embed = umap.UMAP(
    n_components=2,
    n_neighbors=20,
    min_dist=0.1,
    metric="euclidean",
    random_state=SEED,
)
X_embed_umap = reducer_embed.fit_transform(X_embed)

print("UMAP on synthetic embeddings done.")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
palette = sns.color_palette("tab10", n_topics)

for idx, name in enumerate(topic_names):
    mask = y_embed == idx
    ax.scatter(
        X_embed_umap[mask, 0], X_embed_umap[mask, 1],
        s=18, alpha=0.7, color=palette[idx],
        label=name,
    )

ax.set_title(
    "UMAP on synthetic 50-dim embeddings\n"
    "(simulating document/word embedding clusters)",
    fontsize=12, fontweight="bold"
)
ax.set_xticks([])
ax.set_yticks([])
ax.legend(title="Topic cluster", loc="best", fontsize=9, title_fontsize=10)
plt.tight_layout()
plt.savefig("lectures/img/umap_text_embeddings.png", dpi=120, bbox_inches="tight")
plt.show()

**Observations:**
- UMAP clearly recovers the 6 underlying topic clusters even in a 50-dimensional space.
- Clusters that are closer in the original space (lower cluster centre distance) appear
  closer in the 2D projection, demonstrating that **global structure is preserved**.
- In real-world applications (Word2Vec, GloVe, BERT embeddings), UMAP produces
  similarly compact semantic clusters from high-dimensional vectors.

---
## 5. Supervised UMAP

UMAP can incorporate label information to pull same-class points together.  
Pass `y=labels` to `fit_transform` — internally UMAP blends the unsupervised graph with a supervised kNN graph built from the labels.

In [ ]:
reducer_sup = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    random_state=SEED,
)
X_supervised = reducer_sup.fit_transform(X_scaled, y=y)  # pass labels

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
plot_embedding(axes[0], X_umap,       y, "Unsupervised UMAP")
plot_embedding(axes[1], X_supervised, y, "Supervised UMAP")

handles, labels_legend = axes[1].get_legend_handles_labels()
fig.legend(handles, labels_legend, title="Digit",
           loc="lower center", ncol=10, frameon=False,
           bbox_to_anchor=(0.5, -0.02))

fig.suptitle("Unsupervised vs Supervised UMAP on MNIST digits", fontsize=13)
plt.tight_layout()
plt.savefig("lectures/img/umap_supervised.png", dpi=120, bbox_inches="tight")
plt.show()

**Observations:**
- Supervised UMAP produces more clearly separated clusters because it uses class labels to guide the optimisation.
- Useful as a preprocessing step before a classifier (the low-dimensional embedding can be fed directly to a simple classifier).

---
## 6. UMAP for Dimensionality Reduction (not just 2D visualisation)

UMAP is not limited to 2D — it can reduce to any number of dimensions as a preprocessing step.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score

results = {}
dims = [2, 5, 10, 20]

for d in dims:
    emb = umap.UMAP(n_components=d, n_neighbors=15,
                    min_dist=0.1, random_state=SEED).fit_transform(X_scaled)
    knn = KNeighborsClassifier(n_neighbors=5)
    scores = cross_val_score(knn, emb, y, cv=5, scoring="accuracy")
    results[d] = scores.mean()
    print(f"  d={d:2d}  →  kNN-5 accuracy = {scores.mean():.3f} ± {scores.std():.3f}")

# Baseline: kNN on original 64-dim data
baseline = cross_val_score(KNeighborsClassifier(n_neighbors=5),
                           X_scaled, y, cv=5, scoring="accuracy").mean()
print(f"\n  Baseline (64 dims): {baseline:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar([str(d) for d in dims], [results[d] for d in dims],
       color="steelblue", alpha=0.8)
ax.axhline(baseline, color="tomato", linestyle="--", label=f"Original 64-dim ({baseline:.3f})")
ax.set_xlabel("UMAP target dimension")
ax.set_ylabel("5-fold CV accuracy")
ax.set_title("kNN accuracy after UMAP reduction (MNIST digits)")
ax.legend(fontsize=9)
ax.set_ylim(0.9, 1.01)
plt.tight_layout()
plt.savefig("lectures/img/umap_classification.png", dpi=120, bbox_inches="tight")
plt.show()

**Observation:** Even a 10-dimensional UMAP embedding achieves accuracy close to or matching the full 64-dimensional representation, demonstrating that UMAP is a powerful preprocessing tool — not just for visualisation.

---
## 7. UMAP vs t-SNE — Summary Comparison

| Property | t-SNE | UMAP |
|---|---|---|
| **Speed** | $O(N \log N)$ with BH tree | $O(N^{1.14})$ — faster in practice |
| **Global structure** | Poorly preserved | Better preserved |
| **Determinism** | Sensitive to initialisation | More stable (spectral init) |
| **Out-of-sample** | Requires re-running | `transform()` available |
| **Supervised mode** | Not natively supported | Yes (`y=labels`) |
| **Target dimension** | Typically 2–3 only | Any $d$ |
| **Theoretical basis** | KL divergence on probability distributions | Riemannian geometry + algebraic topology |

**When to use UMAP:**
- Large datasets (tens of thousands of points or more)
- When you need to project new points after fitting
- When global structure matters (e.g. trajectory / manifold analysis)
- As a preprocessing step before classification or clustering

---
## References

1. McInnes, L., Healy, J., & Melville, J. (2018). **UMAP: Uniform Manifold Approximation and Projection for Dimension Reduction**. *arXiv:1802.03426*. https://arxiv.org/abs/1802.03426
2. McInnes, L., Healy, J., Saul, N., & Großberger, L. (2018). **UMAP: Uniform Manifold Approximation and Projection**. *Journal of Open Source Software*, 3(29), 861.
3. van der Maaten, L., & Hinton, G. (2008). **Visualizing Data using t-SNE**. *Journal of Machine Learning Research*, 9, 2579–2605.
4. UMAP documentation: https://umap-learn.readthedocs.io/